# ① 安装 KataGo 优化版引擎：环境、优化引擎、权重

在 Linux 发布包解压目录打开此 Notebook，按顺序运行。支持 Ubuntu 22.04/24.04 x86_64、SM120 GPU（包括 RTX 50 / RTX PRO 6000 Blackwell）。

默认安装 CUDA 13.2、cuDNN 9.24.x，以及本发布包的优化引擎。首次安装从 [官方模型页](https://katagotraining.org/networks/) 读取 **Strongest confidently-rated network** 并下载原始权重。下载保留模型原样，记录 SHA256，完成 GPU 出招检查后才标记安装成功。

此文件不安装 iKataGo、不设置账号。完成后打开 `ikatago install.ipynb`。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PACKAGE_DIR = Path.cwd()                  # Linux 发布包解压目录
ENGINE_DIR = Path.home() / "katago-rtx50" # 独立引擎安装目录
WORK_DIR = Path.home() / "work"           # 下一步 iKataGo 安装目录
INSTALL_ENV = True                       # 已装好 CUDA/cuDNN 时可以改为 False
LOCAL_MODEL = ""                         # 留空：下载官方最强评级权重；或填本地模型完整路径
STATE = Path.home() / ".config/katago-rtx50/notebook.json"

if sys.platform != "linux":
    raise RuntimeError("请在 Linux 云机运行。")
PACKAGE_DIR = PACKAGE_DIR.expanduser().resolve()
ENGINE_DIR = ENGINE_DIR.expanduser().absolute()
WORK_DIR = WORK_DIR.expanduser().absolute()
if not (PACKAGE_DIR / "katago").is_file() or not (PACKAGE_DIR / "prepare_katago.py").is_file():
    raise RuntimeError("请把 PACKAGE_DIR 改为 Linux 发布包解压目录。")


## 安装环境

In [ ]:
if INSTALL_ENV:
    command = ["bash", str(PACKAGE_DIR / "setup-linux.sh")]
    if os.geteuid() != 0:
        command = ["sudo", "-n"] + command
    subprocess.run(command, check=True)
else:
    print("保留现有 CUDA/cuDNN 环境。")


## 安装引擎和模型

已有本工具创建的完整安装会校验并保留；不会重复下载或自动换权重。要重新选择当前最强模型，可指定新的 ENGINE_DIR。不要把 ENGINE_DIR 与 WORK_DIR 设置为同一个目录。


In [ ]:
if ENGINE_DIR.resolve() == WORK_DIR.resolve():
    raise RuntimeError("ENGINE_DIR 和 WORK_DIR 必须不同。")
command = [sys.executable, str(PACKAGE_DIR / "prepare_katago.py"), "--destination", str(ENGINE_DIR)]
if LOCAL_MODEL:
    command += ["--model", str(Path(LOCAL_MODEL).expanduser().resolve())]
subprocess.run(command, check=True)

# 保存路径供另外两份 Notebook 读取；不含任何账号密码。
STATE.parent.mkdir(parents=True, exist_ok=True)
temporary = STATE.with_suffix(".tmp")
temporary.write_text(json.dumps({"engineDir": str(ENGINE_DIR), "workDir": str(WORK_DIR)}, indent=2) + "\n")
temporary.chmod(0o600)
temporary.replace(STATE)
prepared = json.loads((ENGINE_DIR / "prepared.json").read_text())
print("引擎已安装：", ENGINE_DIR / "katago")
print("权重已安装：", ENGINE_DIR / prepared["model"])
print("下一步：打开 ikatago install.ipynb")
